# 14 Supervisor Diagnostic Strategy


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

TABLES_DIR = PROJECT_ROOT / 'outputs' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

from run_supervisor_diagnostic_experiments import run_pipeline


## 1. Scientific Framing


## 2. Mathematical Problem Definition


## 3. Dataset Summary Table


In [ ]:
outputs = run_pipeline(project_root=str(PROJECT_ROOT))
dataset_summary = outputs['supervisor_dataset_summary']
news_shift_results = outputs['supervisor_news_shift_results']
per_stock_results = outputs['supervisor_per_stock_results']
correlation_diagnostics = outputs['supervisor_correlation_diagnostics']
target_comparison = outputs['supervisor_target_comparison']

dataset_summary


## 4. Experiment A: News Time-Shift Diagnostic


In [ ]:
news_shift_results.sort_values('roc_auc', ascending=False).head(14)


In [ ]:
plot_df = (
    news_shift_results
    .groupby(['news_day_offset', 'model_name'], as_index=False)['roc_auc']
    .max()
    .sort_values('news_day_offset')
)

fig, ax = plt.subplots(figsize=(8, 4.5))
for model_name, model_df in plot_df.groupby('model_name'):
    ax.plot(model_df['news_day_offset'], model_df['roc_auc'], marker='o', label=model_name)
ax.axhline(0.5, color='black', linewidth=1, linestyle='--')
ax.axvline(0, color='grey', linewidth=1, linestyle=':')
ax.set_title('News Time-Shift Diagnostic')
ax.set_xlabel('News day offset relative to stock return day')
ax.set_ylabel('ROC AUC')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'supervisor_news_shift_roc_auc.png', dpi=200)
plt.show()


## 5. Experiment B: Per-Stock Diagnostics


In [ ]:
per_stock_results.sort_values(['target_name', 'slice_name', 'ticker', 'feature_set']).head(40)


In [ ]:
stock_plot = per_stock_results[
    (per_stock_results['target_name'] == 'next_day_excess_gt_0') &
    (per_stock_results['slice_name'] == 'news_days')
].copy()

pivot = stock_plot.pivot_table(index='ticker', columns='feature_set', values='roc_auc', aggfunc='max')
pivot = pivot.sort_values('price_plus_finbert', ascending=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
pivot.plot(kind='bar', ax=ax)
ax.axhline(0.5, color='black', linewidth=1, linestyle='--')
ax.set_title('Per-Stock News-Day Performance')
ax.set_xlabel('Ticker')
ax.set_ylabel('ROC AUC')
ax.legend(title='Feature set')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'supervisor_per_stock_news_days_roc_auc.png', dpi=200)
plt.show()

pivot


## 6. Experiment C: Target Comparison


In [ ]:
target_comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
target_comparison.sort_values('mean_roc_auc').plot(
    kind='barh',
    x='target_name',
    y='mean_roc_auc',
    ax=ax,
    legend=False,
)
ax.axvline(0.5, color='black', linewidth=1, linestyle='--')
ax.set_title('Best Walk-Forward Result by Target')
ax.set_xlabel('Mean ROC AUC')
ax.set_ylabel('Target')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'supervisor_target_comparison_roc_auc.png', dpi=200)
plt.show()


## 7. Experiment D: Correlation Diagnostics


In [ ]:
correlation_diagnostics.head(25)


In [ ]:
heat_df = correlation_diagnostics.pivot_table(
    index='feature',
    columns='ticker',
    values='corr_with_next_day_excess_return',
)

fig, ax = plt.subplots(figsize=(9, 4.8))
im = ax.imshow(heat_df.fillna(0), aspect='auto', cmap='coolwarm', vmin=-0.5, vmax=0.5)
ax.set_xticks(range(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns)
ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index)
ax.set_title('Sentiment Feature Correlation with Next-Day Excess Return')
fig.colorbar(im, ax=ax, label='Correlation')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'supervisor_sentiment_correlation_heatmap.png', dpi=200)
plt.show()
